# Äquivalenzvergleich: Zeitexpandierter MILP-Solver vs. Dijkstra-Router

Dieses Notebook vergleicht die Ergebnisse des zeitexpandierten MILP-Solvers mit dem implementierten `DijkstraRouter` für verschiedene Gewichtungen und Sendungsszenarien auf `dataset/multimodal_network.json`.

## Hintergrund

**Einzelne Sendung:** Da für eine einzelne Sendung das Problem der optimalen Routenwahl ein klassisches Kürzeste-Weg-Problem auf dem zeitexpandierten Graphen darstellt, müssen sowohl der MILP-Solver als auch der Dijkstra-Router exakt dasselbe Ergebnis liefern.

---

## 1. Setup und Imports

In [9]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from freight_routing.data_loader import NetworkDataLoader
from freight_routing.data_models import Shipment, ObjectiveWeights
from freight_routing.model import TimeExpandedFreightRoutingModel
from heuristics.dijkstra_router import DijkstraRouter
from freight_routing.visualization import create_network_map

## 2. Laden des Netzwerks

In [10]:
network_path = PROJECT_ROOT / "dataset/multimodal_network.json"
print(f"Loading network from {network_path}...")
network_data = NetworkDataLoader.from_json(network_path)
network_data.summary()

Loading network from /home/benedikt/Projects/Sustainable_Freight_Mode_Choice/dataset/multimodal_network.json...
Summary NetworkData:
hubs=100
arcs=1664
modes=4


## 3. Definition einer einzelnen Sendung für den ersten Test

In [11]:
shipment_1 = Shipment(
    id="user_shipment_1",
    start_hub="ALG_185",
    end_hub="ORA_186",
    start_time=0,
    deadline=2880,  # 2 Tage Horizont
    max_price=1_000_000.0,
    max_emissions=None,
    weight=2.0,
)

## 4. Basis-Äquivalenztest (Kostenminimierung)

In [6]:
weights_cost = ObjectiveWeights(cost=1.0, emissions=0.0, time=0.0)

# MILP
milp_model = TimeExpandedFreightRoutingModel(
    network_data, objective_weights=weights_cost
)
milp_model.build(planning_days=2, shipments=[shipment_1])
milp_res = milp_model.solve([shipment_1])

# Dijkstra
dijkstra_router = DijkstraRouter(network_data, objective_weights=weights_cost)
dijkstra_res = dijkstra_router.solve(shipment_1, planning_days=2)

print("--- Basis-Kostenoptimierung ---")
print(
    f"MILP Cost: {milp_res.total_cost:.2f} EUR | Dijkstra Cost: {dijkstra_res.total_cost:.2f} EUR"
)
milp_route = milp_res.shipment_routes[shipment_1.id]
dijkstra_route = dijkstra_res.shipment_routes[shipment_1.id]
milp_path = " -> ".join(f"{arc.from_node.hub_id}({arc.mode})" for arc in milp_route)
dijkstra_path = " -> ".join(
    f"{arc.from_node.hub_id}({arc.mode})" for arc in dijkstra_route
)
print(f"Pfade identisch: {milp_path == dijkstra_path}")

--- Basis-Kostenoptimierung ---
MILP Cost: 1148.64 EUR | Dijkstra Cost: 1148.64 EUR
Pfade identisch: True


## 5. Äquivalenztest mit unterschiedlichen Zielfunktions-Gewichten

1. **Reine Kostenoptimierung**
2. **Reine Emissionsoptimierung**
3. **Reine Zeitoptimierung**
4. **Ausgewogene Optimierung**

**Hinweis zu Einzelszenarien:** Bei reinen Einzelszenarien (z. B. nur Emissions- oder Zeitoptimierung) kann es theoretisch mehrere mathematisch gleichwertige Pfade geben. Da z.B. das Warten an Hubs 0 CO2-Emissionen verursachen kann, ist jeder Abfahrtszeitpunkt für die Emissionen gleichermaßen optimal. Dijkstra wählt den frühesten Start, während der MILP-Solver einen beliebigen gleichwertigen Pfad wählt.

In [ ]:
scenarios = [
    ("Reine Kostenoptimierung", ObjectiveWeights(cost=1.0, emissions=0.0, time=0.0)),
    ("Reine Emissionsoptimierung", ObjectiveWeights(cost=0.0, emissions=1.0, time=0.0)),
    ("Reine Zeitoptimierung", ObjectiveWeights(cost=0.0, emissions=0.0, time=1.0)),
    ("Ausgewogene Optimierung", ObjectiveWeights(cost=0.4, emissions=0.4, time=0.2)),
]

for name, scenario_weights in scenarios:
    print(f"\n=== Szenario: {name} ===")

    # MILP
    m_model = TimeExpandedFreightRoutingModel(
        network_data, objective_weights=scenario_weights
    )
    m_model.build(planning_days=2, shipments=[shipment_1])
    m_res = m_model.solve([shipment_1])

    # Dijkstra
    d_router = DijkstraRouter(network_data, objective_weights=scenario_weights)
    d_res = d_router.solve(shipment_1, planning_days=2)

    if m_res.is_optimal and d_res.is_optimal:
        m_route = m_res.shipment_routes[shipment_1.id]
        d_route = d_res.shipment_routes[shipment_1.id]
        m_path = " -> ".join(f"{arc.from_node.hub_id}({arc.mode})" for arc in m_route)
        d_path = " -> ".join(f"{arc.from_node.hub_id}({arc.mode})" for arc in d_route)

        cost_diff = abs(m_res.total_cost - d_res.total_cost)
        time_diff = abs(m_res.total_time - d_res.total_time)
        emissions_diff = abs(m_res.total_emissions - d_res.total_emissions)
        obj_diff = (
            abs(m_res.objective_value - d_res.objective_value)
            if (m_res.objective_value is not None and d_res.objective_value is not None)
            else 0.0
        )

        print(
            f"  Kosten: MILP={m_res.total_cost:.2f} EUR | Dijkstra={d_res.total_cost:.2f} EUR"
        )
        print(
            f"  CO2:    MILP={m_res.total_emissions:.2f} kg  | Dijkstra={d_res.total_emissions:.2f} kg"
        )
        print(
            f"  Dauer:  MILP={m_res.total_time:.2f} min | Dijkstra={d_res.total_time:.2f} min"
        )
        print(f"  Pfade identisch: {m_path == d_path}")

        if "Kosten" in name:
            assert cost_diff < 1e-3, f"Kosten weichen ab: {cost_diff}"
        elif "Emission" in name:
            assert emissions_diff < 1e-3, f"Emissionen weichen ab: {emissions_diff}"
        elif "Zeit" in name:
            assert time_diff < 1e-3, f"Dauer weicht ab: {time_diff}"
        else:
            assert obj_diff < 1e-3, f"Zielfunktionswert weicht ab: {obj_diff}"

        print(f"  [ERFOLG] Äquivalenz verifiziert!")
    else:
        print("  [FEHLER] Einer der Algorithmen fand keine Lösung.")


=== Szenario: Reine Kostenoptimierung ===
  Kosten: MILP=1148.64 EUR | Dijkstra=1148.64 EUR
  CO2:    MILP=104.90 kg  | Dijkstra=104.90 kg
  Dauer:  MILP=281.00 min | Dijkstra=281.00 min
  Pfade identisch: True
  [ERFOLG] Äquivalenz verifiziert!

=== Szenario: Reine Emissionsoptimierung ===
  Kosten: MILP=1188.64 EUR | Dijkstra=1148.64 EUR
  CO2:    MILP=104.90 kg  | Dijkstra=104.90 kg
  Dauer:  MILP=521.00 min | Dijkstra=281.00 min
  Pfade identisch: False
  [ERFOLG] Äquivalenz verifiziert!

=== Szenario: Reine Zeitoptimierung ===
  Kosten: MILP=1148.64 EUR | Dijkstra=1148.64 EUR
  CO2:    MILP=104.90 kg  | Dijkstra=104.90 kg
  Dauer:  MILP=281.00 min | Dijkstra=281.00 min
  Pfade identisch: True
  [ERFOLG] Äquivalenz verifiziert!

=== Szenario: Ausgewogene Optimierung ===
  Kosten: MILP=1148.64 EUR | Dijkstra=1148.64 EUR
  CO2:    MILP=104.90 kg  | Dijkstra=104.90 kg
  Dauer:  MILP=281.00 min | Dijkstra=281.00 min
  Pfade identisch: True
  [ERFOLG] Äquivalenz verifiziert!
